In [2]:
import os
import json
from typing import List
from pathlib import Path
from dotenv import load_dotenv
from llama_index.readers.file import PyMuPDFReader
from llama_index.core import Document, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configuration
load_dotenv(dotenv_path=".env")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY can't be found"

LLM_MODEL = "gpt-4o-mini"
EMB_MODEL = "text-embedding-3-small"
PDF_PATH = "input/00zentai.pdf"
JSONL_PATH = "input/data.jsonl"
PERSIST_DIR = "storage"

# Initialize models
llm = OpenAI(model=LLM_MODEL)
embed = OpenAIEmbedding(model=EMB_MODEL)

def load_pdf_documents(pdf_path: str) -> List[Document]:
    """Load PDF documents using PyMuPDFReader."""
    pdf_reader = PyMuPDFReader()
    return pdf_reader.load(file_path=pdf_path)

def load_json_as_documents(jsonl_path: str) -> List[Document]:
    """Load JSONL data as documents."""
    documents = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            q = data.get("Question", "")
            a = data.get("Answer", "")
            documents.append(Document(text=f"Q: {q}\nA: {a}", metadata={"url": data.get("url", "")}))
    return documents

def create_or_load_index(pdf_docs: List[Document], jsonl_docs: List[Document]) -> VectorStoreIndex:
    """Create new index or load existing one."""
    all_docs = pdf_docs + jsonl_docs
    
    if not Path(PERSIST_DIR).exists():
        print("Building VectorStoreIndex...")
        index = VectorStoreIndex.from_documents(all_docs, embed_model=embed)
        index.storage_context.persist(PERSIST_DIR)
        print(f"Index saved to {PERSIST_DIR}")
    else:
        print(f"Loading index from {PERSIST_DIR}")
        storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
        index = load_index_from_storage(storage_context)
    
    return index

# Load documents
pdf_docs = load_pdf_documents(PDF_PATH)
jsonl_docs = load_json_as_documents(JSONL_PATH)
print(f"Loaded {len(pdf_docs)} PDF docs and {len(jsonl_docs)} JSONL docs")

# Create/load index and query engine
index = create_or_load_index(pdf_docs, jsonl_docs)
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=10,
    vector_store_query_mode="mmr",
)
print("Query engine ready")

async def run_questions(qengine, questions: List[str]):
    """Run a list of questions through the query engine."""
    for i, question in enumerate(questions, 1):
        print(f"Question {i}: {question}")
        response = await qengine.aquery(question)
        print(f"Answer: {str(response)[:500]}...\n")

# Sample questions
questions = [
    "信書便事業の趣旨と国の取組は何ですか？",
    "会計検査院は独立行政法人等をどのように検査していますか？",
    "日本のICTインフラ整備に関する白書の記述を要約してください。",
    "マイナンバーの利用目的はどのように明示すればよいですか？",
    "デジタル田園都市国家構想の柱を、白書の記述に基づいて挙げてください。",
]

# Run questions
await run_questions(query_engine, questions)


Loaded 320 PDF docs and 22794 JSONL docs
Loading index from storage
Loading llama_index.core.storage.kvstore.simple_kvstore from storage/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from storage/index_store.json.
Query engine ready
Question 1: 信書便事業の趣旨と国の取組は何ですか？
Answer: 信書便事業は、信書の安全かつ迅速な配送を目的とした事業であり、国はこの事業を通じて、通信の自由や信書の秘密を保障することを目指しています。具体的には、信書便事業者に対する規制や支援を行い、信書の適切な取り扱いや配送の信頼性を確保するための取り組みを進めています。また、信書便事業の普及を促進し、利用者の利便性向上を図るための施策も実施されています。...

Question 2: 会計検査院は独立行政法人等をどのように検査していますか？
Answer: 会計検査院は、独立行政法人等に対して、財務状況や業務の適正性を検査するために、定期的な監査を行います。この監査では、財務諸表の確認や業務運営の効率性、法令遵守の状況などが評価されます。また、必要に応じて特別な調査を実施し、検査結果に基づいて改善提言を行うこともあります。...

Question 3: 日本のICTインフラ整備に関する白書の記述を要約してください。
Answer: 日本のICTインフラ整備に関する白書では、近年の通信技術の進展やデジタル化の進展に伴い、ICTインフラの重要性が増していることが強調されています。特に、通信網の整備やデータセンターの拡充が進められ、地域間のデジタル格差を解消するための取り組みが行われています。また、サイバーセキュリティの強化や、持続可能な社会の実現に向けたICTの活用も重要なテーマとして取り上げられています。これらの施策は、経済成長や社会の発展に寄与することを目指しています。...

Question 4: マイナンバーの利用目的はどのように明示すればよいですか？
Answer: マイナンバ

In [ ]:
%pip install PyMuPDF